# Settimana 22 — Comunicazione di progetti IA a clienti non tecnici

**Obiettivo:** generare i tre documenti di consegna per la ResearchAgent:
README tecnico, guida d'uso e sintesi esecutiva.
Inoltre, costruire un generatore riutilizzabile che tu possa usare in qualsiasi progetto futuro.

**Al termine avrai** in `proyectos/research-agent/`:
- `docs/README_tecnico.md` — documentazione per IT
- `docs/guia_uso.md` — per il team che usa il sistema
- `docs/resumen_ejecutivo.md` — per la direzione
- `docs/prompts/clasificador_v1.md` — template di documentazione del prompt
- Generatore di sintesi esecutiva con LLM a partire da metriche reali

**Costo stimato:** < $0.02 (un paio di chiamate a gpt-4o-mini per la generazione di testo).

## Cella 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"])

import os, json
from pathlib import Path
from datetime import date
from dotenv import load_dotenv

PROJECT_DIR = Path("../proyectos/research-agent").resolve()
load_dotenv(PROJECT_DIR / ".env")
sys.path.insert(0, str(PROJECT_DIR))

from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
assert os.getenv("OPENAI_API_KEY"), "Definisci OPENAI_API_KEY nel .env"

DOCS_DIR    = PROJECT_DIR / "docs"
PROMPTS_DIR = DOCS_DIR / "prompts"
DOCS_DIR.mkdir(exist_ok=True)
PROMPTS_DIR.mkdir(exist_ok=True)

HOY = date.today().strftime("%Y-%m-%d")
print(f"✓ Progetto: {PROJECT_DIR}")
print(f"✓ Directory docs: {DOCS_DIR}")
print(f"✓ Data: {HOY}")

## Cella 1 — README tecnico

Documento per il team IT del cliente: come distribuire, mantenere e fare debug.

In [ ]:
README_TECNICO = f"""# ResearchAgent API — Documentazione tecnica

**Versione:** 1.0
**Ultimo aggiornamento:** {HOY}
**Mantenuto da:** [Il tuo nome] — [tu@email.com]

---

## Architettura

```
Client HTTP
    │
    ▼
FastAPI (api.py)
    │  ├── security/guardrails.py   ← validazione input/output
    │  ├── security/rate_limiter.py ← controllo abusi
    │  └── jobs.py (JobStore)       ← stato in memoria
    │
    ▼ asyncio.create_task
ResearchAgent (agent/agent.py)
    │  ├── planner.py    ← genera il piano di ricerca
    │  ├── executor.py   ← esegue gli strumenti nel ciclo ReAct
    │  │     ├── tools.py → buscar_web (DuckDuckGo, senza chiave)
    │  │     ├── tools.py → leer_url (BeautifulSoup)
    │  │     └── tools.py → buscar_en_corpus (Chroma RAG)
    │  └── synthesizer.py ← genera il report finale
    │
    ▼
observability/ (Tracer + Logger + Langfuse opzionale)
```

## Requisiti di sistema

- Python 3.11+
- 512 MB RAM minimo (1 GB consigliato)
- Accesso a internet (per DuckDuckGo e OpenAI API)
- Porta 8000 libera (configurabile)

## Variabili d'ambiente

Crea un file `.env` nella root del progetto:

```bash
# Obbligatorio
OPENAI_API_KEY=sk-...

# Opzionale — Langfuse (osservabilità)
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com

# Opzionale — configurazione
MAX_PASOS_AGENTE=6         # numero massimo di passi del ciclo ReAct
MODELO_LLM=gpt-4o-mini     # modello da usare
```

## Come distribuire

### Locale
```bash
pip install -r requirements.txt
uvicorn api:app --host 0.0.0.0 --port 8000
```

### Docker
```bash
docker build -t research-agent .
docker run -p 8000:8000 --env-file .env research-agent
```

### Railway / Render
1. Collega il repository nel pannello di Railway o Render
2. Definisci le variabili d'ambiente nel pannello di configurazione
3. Il deploy è automatico a ogni push su `main`

## Come eseguire i test

```bash
# Test rapidi (senza chiamate a OpenAI)
pytest test_api.py test_security.py -v

# Test di integrazione (richiedono OPENAI_API_KEY)
pytest -m lento -v
```

## Come monitorare

### Health check
```bash
curl http://localhost:8000/health
# Risposta attesa: {{"status": "ok", "jobs_activos": 0}}
```

### Log
- I log strutturati (JSON) vengono scritti in `logs/research_agent.log`
- Ogni chiamata all'LLM registra: token usati, costo stimato, latenza
- Gli errori includono il `job_id` per la correlazione

### Metriche su Langfuse (se configurato)
- Dashboard su `https://cloud.langfuse.com` → progetto `research-agent`
- Metriche disponibili: costo per traccia, latenza p95, tasso di errore

## Come aggiornare i prompt

I prompt si trovano in `agent/synthesizer.py` (funzione `generar_informe`) e
`agent/planner.py` (funzione `crear_plan`).
Se usi Langfuse, puoi anche gestirne le versioni dal pannello web senza toccare il codice.

## Troubleshooting

| Errore | Causa probabile | Soluzione |
|-------|---------------|----------|
| `AuthenticationError` all'avvio | `OPENAI_API_KEY` configurata male | Verifica il `.env` e riavvia |
| Job sempre in stato `pending` | Il server non è partito con `asyncio` | Usa `uvicorn`, non `python api.py` |
| `RateLimitError` di OpenAI | Troppe richieste concorrenti | Riduci `MAX_PASOS_AGENTE` o aggiungi `time.sleep` tra le chiamate |
| Risposte molto lente (>30s) | Throttling di DuckDuckGo | Normale; l'agente riprova automaticamente |
| `JobNotFound 404` interrogando lo stato | Il job è scaduto (riavvio del server) | I job sono salvati in memoria; riavvia il job |
"""

(DOCS_DIR / "README_tecnico.md").write_text(README_TECNICO)
print("✓ docs/README_tecnico.md generato")
print(f"  {len(README_TECNICO.splitlines())} righe")

## Cella 2 — Guida d'uso

Per il team che userà il sistema quotidianamente.

In [ ]:
GUIA_USO = f"""# ResearchAgent — Guida d'uso

**Versione:** 1.0
**Ultimo aggiornamento:** {HOY}

---

## Cosa fa questo sistema

ResearchAgent cerca informazioni su internet e nella vostra base di documenti su
qualsiasi tema gli indichiate, e genera un report strutturato in pochi minuti.
Quello che prima richiedeva 30-60 minuti di ricerca manuale ora richiede tra 1 e 3 minuti.

---

## Come lanciare una ricerca

### Opzione A — Dal client web (se disponibile)
1. Apri l'URL del sistema
2. Scrivi il tema da ricercare
3. Clicca su **Investigar**
4. Attendi 1-3 minuti e scarica il report

### Opzione B — Dall'API
```bash
# Lancia una ricerca
curl -X POST http://localhost:8000/research \\
  -H 'Content-Type: application/json' \\
  -d '{{"tema": "Applicazioni dell'IA nell'acquacoltura galiziana 2025"}}'

# Risposta: {{"job_id": "a1b2c3d4", "status": "pending", ...}}

# Controlla lo stato
curl http://localhost:8000/jobs/a1b2c3d4

# Scarica il report (quando status = completed)
curl http://localhost:8000/jobs/a1b2c3d4/report
```

---

## Casi in cui funziona bene

✓ Temi con presenza su internet (notizie, report, pubblicazioni accademiche recenti)
✓ Domande sullo stato dell'arte: *"Ultime tecnologie di rilevamento delle microplastiche nella pesca"*
✓ Confronti tra tecnologie o fornitori
✓ Sintesi della normativa europea recente
✓ Temi per cui avete documentazione interna caricata nel corpus

## Casi in cui NON funziona bene

✗ **Informazioni riservate interne** non presenti nel corpus: l'agente non accede ai vostri sistemi interni.
✗ **Dati in tempo reale** (prezzi del mercato ittico oggi, meteo ora): il sistema non ha accesso a feed live.
✗ **Temi molto recenti** (ultime 48 ore): DuckDuckGo può impiegare tempo a indicizzarli.
✗ **Domande con risposta unica ed esatta** (*"Quanti chili di nasello sono stati sbarcati martedì?"*): per questo dovete consultare direttamente il vostro sistema gestionale.

---

## Come interpretare il report

Ogni report include:

- **Sintesi esecutiva**: due o tre paragrafi con le idee principali
- **Risultati per area**: suddivisione per sottotemi
- **Fonti consultate**: lista degli URL visitati dall'agente
- **Limiti del report**: cosa non è stato possibile trovare o verificare

**Importante:** il report è un punto di partenza, non una verità assoluta.
Verifica sempre i dati chiave prima di usarli per una decisione importante.

---

## Cosa fare se il risultato sembra sbagliato

1. **Leggi le fonti**: gli URL in fondo al report ti permettono di verificare ogni dato direttamente.
2. **Riformula la domanda**: domande più concrete producono report più precisi. *"Normativa europea pesca sostenibile 2024"* è meglio di *"normativa pesca"*.
3. **Aggiungi contesto**: *"...focalizzato sul settore conserviero galiziano"* aiuta l'agente a filtrare i risultati rilevanti.
4. **Segnala il caso**: se il report contiene un errore fattuale evidente, invialo a [tu@email.com] indicando il job_id. Questi casi servono a migliorare il sistema.

---

## Limiti d'uso

- Massimo **10 ricerche all'ora** per utente
- Lunghezza minima del tema: 10 caratteri
- Lunghezza massima del tema: 2.000 caratteri
- Se ricevi l'errore `429 Too Many Requests`, attendi qualche minuto prima di lanciare un'altra ricerca

---

## Contatti e supporto

- **Supporto tecnico:** [tu@email.com]
- **Tempo di risposta:** 24h nei giorni lavorativi
- **Per segnalare bug:** includi sempre il `job_id` del report interessato
"""

(DOCS_DIR / "guia_uso.md").write_text(GUIA_USO)
print("✓ docs/guia_uso.md generato")
print(f"  {len(GUIA_USO.splitlines())} righe")

## Cella 3 — Metriche simulate del progetto

Prima di generare la sintesi esecutiva, definisci le metriche reali del progetto.
In un progetto reale, queste verrebbero dai log di Langfuse o dal tracer locale.

In [ ]:
# Metriche del progetto (in un progetto reale provengono da logs/Langfuse)
METRICAS_PROYECTO = {
    "cliente": "ANFACO-CYTMA",
    "sector": "industria conserviera ittica galiziana",
    "problema_original": (
        "Il team innovazione dedicava 2-3 ore settimanali a ricerche manuali "
        "di report sulle tendenze tecnologiche nel settore ittico e dell'acquacoltura."
    ),
    "solucion": (
        "Agente di ricerca autonomo che genera report strutturati in 1-3 minuti "
        "cercando su internet e nella base documentale interna del cliente."
    ),
    # Metriche quantitative
    "informes_generados_mes": 87,
    "tiempo_medio_informe_s": 94,
    "tiempo_manual_estimado_min": 45,
    "coste_por_informe_eur": 0.008,
    "latencia_p95_s": 142,
    "tasa_completados": 0.96,
    # Limitazioni oneste
    "limitaciones": [
        {
            "descripcion": "I temi pubblicati da meno di 48 ore potrebbero non comparire nei risultati",
            "impacto_porcentaje": 4,
            "mitigacion": "Il sistema contrassegna questi report per la revisione manuale"
        },
        {
            "descripcion": "Non accede a banche dati a pagamento (EUMOFA, Globefish)",
            "impacto_porcentaje": 15,
            "mitigacion": "Si può ampliare aggiungendo connettori specifici in fase 2"
        },
    ]
}

# Calcola metriche derivate
horas_ahorradas_mes = (
    METRICAS_PROYECTO["informes_generados_mes"] *
    METRICAS_PROYECTO["tiempo_manual_estimado_min"] / 60
)
coste_total_mes = (
    METRICAS_PROYECTO["informes_generados_mes"] *
    METRICAS_PROYECTO["coste_por_informe_eur"]
)

print("Metriche del progetto:")
print(f"  Report generati/mese:   {METRICAS_PROYECTO['informes_generados_mes']}")
print(f"  Tempo medio/report:     {METRICAS_PROYECTO['tiempo_medio_informe_s']}s")
print(f"  Tasso di successo:      {METRICAS_PROYECTO['tasa_completados']*100:.0f}%")
print(f"  Ore risparmiate/mese:   {horas_ahorradas_mes:.0f}h")
print(f"  Costo totale/mese:      {coste_total_mes:.2f}€")
print(f"  Costo per ora risparmiata: {coste_total_mes/horas_ahorradas_mes:.3f}€")

## Cella 4 — Sintesi esecutiva generata con LLM

A partire dalle metriche strutturate, l'LLM genera il testo esecutivo.
Tu definisci i fatti; l'LLM li converte in linguaggio di business.

In [ ]:
def generar_resumen_ejecutivo(metricas: dict) -> str:
    """
    Genera la sintesi esecutiva di un progetto IA a partire da metriche strutturate.
    L'LLM scrive il testo; i fatti li metti tu.
    """
    limitaciones_txt = "\n".join(
        f"- {lim['descripcion']} (interessa ~{lim['impacto_porcentaje']}% dei casi). "
        f"Mitigazione: {lim['mitigacion']}."
        for lim in metricas["limitaciones"]
    )

    prompt = f"""Scrivi la sintesi esecutiva di fine progetto per un cliente aziendale.
Usa un linguaggio diretto, senza gergo tecnico. Massimo 400 parole.
Struttura: paragrafo di contesto → paragrafo di risultati → paragrafo di limitazioni → paragrafo di prossimi passi.

DATI DEL PROGETTO:
- Cliente: {metricas['cliente']} ({metricas['sector']})
- Problema originale: {metricas['problema_original']}
- Soluzione implementata: {metricas['solucion']}

METRICHE (usale tutte, traducile in impatto di business):
- {metricas['informes_generados_mes']} report generati nel primo mese
- Tempo medio per report: {metricas['tiempo_medio_informe_s']} secondi
  (vs. {metricas['tiempo_manual_estimado_min']} minuti di ricerca manuale)
- Tasso di successo: {metricas['tasa_completados']*100:.0f}%
- Costo per report: {metricas['coste_por_informe_eur']:.3f}€
- Latenza p95: {metricas['latencia_p95_s']}s

LIMITAZIONI NOTE (menzionale con onestà usando il framework AIR — riconoscere, impatto, risoluzione):
{limitaciones_txt}

PROSSIMI PASSI CONSIGLIATI: suggerisci 2-3 miglioramenti concreti con il loro impatto atteso."""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=600,
    )
    return resp.choices[0].message.content.strip()


print("Generazione sintesi esecutiva con gpt-4o-mini...")
resumen_llm = generar_resumen_ejecutivo(METRICAS_PROYECTO)

# Intestazione del documento
RESUMEN_EJECUTIVO = f"""# ResearchAgent — Sintesi esecutiva

**Cliente:** {METRICAS_PROYECTO['cliente']}
**Data:** {HOY}
**Preparato da:** [Il tuo nome]

---

{resumen_llm}

---

## Metriche di riferimento

| Metrica | Valore |
|---------|-------|
| Report generati (primo mese) | {METRICAS_PROYECTO['informes_generados_mes']} |
| Tempo medio per report | {METRICAS_PROYECTO['tiempo_medio_informe_s']}s |
| Tempo manuale equivalente | {METRICAS_PROYECTO['tiempo_manual_estimado_min']} min |
| Tasso di successo | {METRICAS_PROYECTO['tasa_completados']*100:.0f}% |
| Costo per report | {METRICAS_PROYECTO['coste_por_informe_eur']:.3f}€ |
| Latenza p95 | {METRICAS_PROYECTO['latencia_p95_s']}s |
"""

(DOCS_DIR / "resumen_ejecutivo.md").write_text(RESUMEN_EJECUTIVO)
print("✓ docs/resumen_ejecutivo.md generado")
print()
print("--- Contenuto generato ---")
print(resumen_llm)

## Cella 5 — Documentazione dei prompt

In [ ]:
# Legge il prompt attuale del synthesizer per documentarlo
SYNTHESIZER_PATH = PROJECT_DIR / "agent" / "synthesizer.py"

# Estratto del prompt di sintesi (hardcoded come fallback in synthesizer.py)
PROMPT_ACTUAL = """Sei un ricercatore esperto. Sulla base della seguente ricerca,
genera un report dettagliato e ben strutturato in spagnolo.

Tema: {tema}

Evidenze raccolte:
{evidencias}

Il report deve includere:
1. Sintesi esecutiva (2-3 paragrafi)
2. Risultati principali organizzati per area tematica
3. Conclusioni e raccomandazioni
4. Limiti della ricerca"""

DOC_PROMPT = f"""# Prompt: Sintetizzatore di report di ricerca

**Nome:** synthesizer_informe
**Versione:** 1.0
**Ultimo aggiornamento:** {HOY}
**Modello:** gpt-4o-mini
**Temperatura:** 0.3

---

## Scopo

Sintetizza le evidenze raccolte dall'agente (ricerche web, URL lette,
corpus interno) in un report strutturato in spagnolo.

## Variabili del prompt

| Variabile | Tipo | Descrizione |
|----------|------|-------------|
| `{{tema}}` | string | Il tema di ricerca richiesto dall'utente |
| `{{evidencias}}` | string | JSON con le evidenze raccolte dall'esecutore |

## Quando modificarlo

- Se il cliente chiede report in un'altra lingua: cambia "in spagnolo" e indica la lingua
- Se serve una sezione aggiuntiva: aggiungila alla lista numerata finale
- Se i report sono troppo lunghi: aggiungi `Massimo N parole per sezione`
- Se il cliente ha un formato aziendale specifico: aggiungi un esempio nella sezione istruzioni

## Prompt attuale

```
{PROMPT_ACTUAL}
```

## Metriche di prestazione

| Metrica | Valore |
|---------|-------|
| Token input (media) | ~1.200 |
| Token output (media) | ~500 |
| Costo per chiamata | ~$0.0003 |
| Latenza media | ~4s |

## Cronologia delle modifiche

- **v1.0** ({HOY}): Versione iniziale. Struttura di 4 sezioni.
"""

(PROMPTS_DIR / "synthesizer_v1.md").write_text(DOC_PROMPT)
print("✓ docs/prompts/synthesizer_v1.md generato")

# Elenca tutti i documenti generati
print()
print("Documenti di consegna generati:")
for f in sorted(DOCS_DIR.rglob("*.md")):
    rel = f.relative_to(PROJECT_DIR)
    lineas = len(f.read_text().splitlines())
    print(f"  {rel}  ({lineas} righe)")

## Cella 6 — Generatore di demo con dati reali del cliente

Prima di una riunione con il cliente, prepara tre casi di dimostrazione:
uno buono, uno limite e uno di fallimento noto.

In [ ]:
CASOS_DEMO = [
    {
        "tipo": "caso_bueno",
        "etiqueta": "✓ Caso standard (mostra per primo)",
        "tema": "Applicazioni della visione artificiale nelle linee di lavorazione delle conserve di pesce",
        "por_que": "Tema con molte pubblicazioni recenti, ben indicizzato, risultato prevedibile",
    },
    {
        "tipo": "caso_limite",
        "etiqueta": "⚠ Caso limite (mostra onestà)",
        "tema": "Normativa galiziana sugli scarichi nei porti da pesca aggiornamento 2025",
        "por_que": "Normativa regionale recente, potrebbe non essere ben indicizzata. L'agente troverà qualcosa ma potrebbe essere incompleto.",
    },
    {
        "tipo": "caso_fallo",
        "etiqueta": "✗ Caso di fallimento noto (dimostra trasparenza)",
        "tema": "Prezzo dell'acciuga al mercato ittico di Burela oggi",
        "por_que": "Dati in tempo reale — il sistema non li ha. Mostralo per dimostrare che conosci i limiti del sistema.",
    },
]

print("Piano di dimostrazione:")
print("=" * 60)
for i, caso in enumerate(CASOS_DEMO, 1):
    print(f"\nCaso {i}: {caso['etiqueta']}")
    print(f"  Tema:      {caso['tema']}")
    print(f"  Perché:    {caso['por_que']}")

print()
print("Struttura della demo (20 min):")
estructura = [
    ("00:00", "02:00", "Il problema che risolviamo (con le parole del cliente)"),
    ("02:00", "08:00", "Demo Caso 1 — caso standard con i suoi dati"),
    ("08:00", "12:00", "Demo Caso 2 — caso limite + come viene gestito"),
    ("12:00", "14:00", "Demo Caso 3 — fallimento noto + trasparenza"),
    ("14:00", "17:00", "Metriche: ore risparmiate, costo, tasso di successo"),
    ("17:00", "20:00", "Domande"),
]
for inicio, fin, desc in estructura:
    print(f"  {inicio}–{fin}  {desc}")

## Cella 7 — Traduttore di metriche tecniche in linguaggio di business

In [ ]:
def traducir_metrica(metrica_tecnica: str, valor, contexto: str = "") -> str:
    """
    Traduce una metrica tecnica in linguaggio di business usando l'LLM.
    Riutilizzabile per qualsiasi progetto.
    """
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Traduci questa metrica tecnica in una frase in linguaggio di business.
Una sola frase, massimo 25 parole. Senza gergo tecnico.
Contesto del progetto: {contexto}
Metrica: {metrica_tecnica} = {valor}"""
        }],
        temperature=0.3,
        max_tokens=60,
    )
    return resp.choices[0].message.content.strip()


METRICAS_A_TRADUCIR = [
    ("accuracy", "94%"),
    ("latenza p95", "142 secondi"),
    ("costo per report", "0.008€"),
    ("tasso di job completati", "96%"),
    ("tempo medio di generazione", "94 secondi"),
]

CONTEXTO = "sistema di generazione automatica di report di ricerca per il settore ittico galiziano"

print("Traduzione di metriche tecniche:")
print("=" * 60)
for metrica, valor in METRICAS_A_TRADUCIR:
    traduccion = traducir_metrica(metrica, valor, CONTEXTO)
    print(f"  Tecnico:  {metrica} = {valor}")
    print(f"  Business: {traduccion}")
    print()

## Sfida — Email di chiusura progetto

Scrivi una funzione `generar_email_cierre(metricas)` che generi l'email che
invii al cliente quando consegni il progetto. L'email deve:

1. Avere un oggetto accattivante che non sia generico (*"Consegna del progetto"* non va bene)
2. Riassumere l'impatto in **due frasi** senza metriche tecniche
3. Elencare i tre deliverable con un link placeholder per ciascuno
4. Includere una call to action chiara per la riunione di presentazione
5. Terminare con una frase che apra la porta alla fase successiva

```python
def generar_email_cierre(metricas: dict) -> str:
    # TODO: chiama gpt-4o-mini con un prompt che includa tutti i punti
    # Tono: professionale ma cordiale, senza anglicismi non necessari
    pass

email = generar_email_cierre(METRICAS_PROYECTO)
print(email)
```

Prova il risultato con i tuoi dati reali del progetto.

## Riepilogo della settimana 22

| Deliverable | Destinatario | A cosa risponde |
|-----------|-----------|-------------|
| `docs/README_tecnico.md` | Team IT | Come distribuisco e mantengo questo sistema? |
| `docs/guia_uso.md` | Utenti finali | Come ne traggo vantaggio senza romperlo? |
| `docs/resumen_ejecutivo.md` | Direzione | È valsa la pena l'investimento? |
| `docs/prompts/synthesizer_v1.md` | Chiunque | Come funziona e come aggiorno i prompt? |
| `generar_resumen_ejecutivo()` | Riutilizzabile | Genera testo esecutivo a partire dalle metriche |
| `traducir_metrica()` | Riutilizzabile | Converte metriche tecniche in linguaggio di business |

La settimana 23 copre **pricing avanzato e gestione di progetti IA**: come stimare lo sforzo,
gestire cambi di ambito e proteggersi contrattualmente.